# Market Analysis â€” Structures, Parsers & Event Hierarchy

Polymarket entities form a hierarchy: **Series > Events > Markets > Tokens**.

This notebook demonstrates the `parsers` module, which extracts structured
data from the free-text `groupItemTitle` field that Polymarket uses for
bracket bounds, directional thresholds, and sports lines.

Every market falls into one of five **event structures**:

| Structure | Description |
|---|---|
| Single-Outcome | One binary Yes/No market |
| negRisk Multi-Outcome | Mutually exclusive outcomes (prices sum to $1) |
| Non-negRisk Multi-Outcome | Independent binary markets grouped together |
| Directional / Counter-Based | Threshold on a counter (up/down arrows) |
| Bracketed | Numeric value carved into ranges (20-29, <20, 580+) |

In [ ]:
!pip install -q polymarket-pandas

In [ ]:
import pandas as pd

from polymarket_pandas import PolymarketPandas
from polymarket_pandas.parsers import (
    classify_event_structure,
    coalesce_end_date_from_title,
    parse_title_bounds,
    parse_title_sports,
)

client = PolymarketPandas()

## Fetch Events with Expanded Markets

We fetch 300 open events and expand their nested `markets` list into
prefixed columns (e.g. `marketsConditionId`, `marketsGroupItemTitle`).

In [ ]:
df = client.get_events(closed=False, limit=300, expand_markets=True)
print(f"{len(df)} rows, {df['id'].nunique()} unique events")
df.head(5)

## Classify Event Structures

`classify_event_structure` assigns one of the five labels to each row,
based on the number of markets per event, the `negRisk` flag, and
title patterns.

In [ ]:
df["structure"] = classify_event_structure(df)
df["structure"].value_counts()

### The Five Structures

- **Single-Outcome** â€” Events with exactly one binary market (Yes/No).
- **negRisk Multi-Outcome** â€” Mutually exclusive outcomes where prices sum to ~$1. The exchange uses a neg-risk adapter contract.
- **Non-negRisk Multi-Outcome** â€” Multiple independent binary markets grouped under one event.
- **Directional / Counter-Based** â€” Markets with up/down arrow titles tracking whether a counter exceeds a threshold.
- **Bracketed** â€” Numeric ranges carved into bins (e.g. "280-299", "<20", "580+").

## Parse Title Bounds

`parse_title_bounds` extracts numeric bracket edges and directional
thresholds from the `marketsGroupItemTitle` string. Returns four
columns: `boundLow`, `boundHigh`, `direction`, `threshold`.

In [ ]:
# Show rows where bounds were parsed
has_bounds = df_bounds["boundLow"].notna()
cols = ["title", "marketsGroupItemTitle", "boundLow", "boundHigh", "direction", "threshold"]
df_bounds.loc[has_bounds, cols].head(15)

## Parse Sports Lines

`parse_title_sports` extracts spread lines and over/under totals from
sports market titles ("Spread -1.5", "O/U 8.5", "Over 2.5").

In [ ]:
# Show rows where a spread or total line was parsed
has_line = df_sports["spreadLine"].notna() | df_sports["totalLine"].notna()
cols = ["title", "marketsGroupItemTitle", "spreadLine", "totalLine", "side"]
df_sports.loc[has_line, cols].head(15)

## Coalesce End Dates from Titles

Polymarket clears `marketsEndDate` after a market resolves, but the
deadline is often encoded in the title string ("April 7", "March 2").
`coalesce_end_date_from_title` parses these labels and fills in the gaps.

In [ ]:
null_before = df["marketsEndDate"].isna().sum()
df["marketsEndDate"] = coalesce_end_date_from_title(df)
null_after = df["marketsEndDate"].isna().sum()

print(f"Null end dates before: {null_before}")
print(f"Null end dates after:  {null_after}")
print(f"Filled {null_before - null_after} rows from title parsing")

## Tags & Series

Tags categorize events. Series group recurring events together
(e.g. weekly prediction contests).

In [ ]:
tags = client.get_tags(limit=20)
tags.head(10)

In [ ]:
series = client.get_series(closed=False, limit=5, expand_events=True)
print(f"{len(series)} rows from {series['id'].nunique()} series")
series.head(10)